Import libraries:

In [12]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

Pull the data from Postgres database:

In [13]:
load_dotenv()

user = os.environ.get('AACT_DB_USER')
password = os.environ.get('AACT_DB_PASSWORD')
host = os.environ.get('AACT_DB_HOST')
port = os.environ.get('AACT_DB_PORT')
dbname = os.environ.get('AACT_DB_NAME')

engine = create_engine(f'postgresql://{user}:{password}@{host}:{port}/{dbname}')

df = pd.read_sql("SELECT COUNT(*) FROM ctgov.studies;", engine)
print(df)

    count
0  598314


Based off of the provided data dictionary (https://aact.ctti-clinicaltrials.org/data_dictionary), select these attributes:
- nct_id: Unique trial identifier
- phase: Trial phase (1, 2, 3, 4)
- enrollment: Target number of participants a trial aims to recruit
- overall_status: Trial completion status
- start_date: Start of trial
- completion_date: End of trial
- agency_class: Trial lead sponsor type
- num_sites: How many locations the trial ran across

In [16]:
df = pd.read_sql("SELECT s.nct_id, s.phase, s.enrollment, s.overall_status, s.start_date, s.completion_date, sp.agency_class AS sponsor_type, COUNT(DISTINCT f.id) AS num_sites \
                FROM ctgov.studies s \
                LEFT JOIN ctgov.sponsors sp \
                    ON s.nct_id = sp.nct_id AND sp.lead_or_collaborator = 'lead' \
                LEFT JOIN ctgov.facilities f \
                    ON s.nct_id = f.nct_id \
                WHERE s.overall_status = 'COMPLETED' \
                AND s.start_date IS NOT NULL \
                AND s.completion_date IS NOT NULL \
                GROUP BY s.nct_id, s.phase, s.enrollment, s.overall_status, \
                        s.start_date, s.completion_date, sp.agency_class;", engine)


In [17]:
df.head()

,nct_id,phase,enrollment,overall_status,start_date,completion_date,sponsor_type,num_sites
0,NCT00000113,PHASE3,469.0,COMPLETED,1997-09-30,2013-09-30,OTHER,4
1,NCT00000114,PHASE3,NaN,COMPLETED,1984-05-31,1987-06-30,NIH,0
2,NCT00000115,PHASE2,NaN,COMPLETED,1990-12-31,1994-06-30,NIH,0
3,NCT00000116,PHASE3,221.0,COMPLETED,1996-05-31,2002-09-30,NIH,1
4,NCT00000117,PHASE3,NaN,COMPLETED,1995-08-31,1997-12-31,NIH,2
